# Ocean sounds: front-end probe

**Question.** Path A's mel front end was carried over from ESC-50: 128 bands from 20 Hz to 5 kHz
with a 1,024-point FFT. The fish here call mostly below 250 Hz, where that layout puts only 17 of
its 128 bands, while 46 bands cover 2-5 kHz, which is mostly snapping shrimp and background noise.
Would a layout matched to these species keep more of what tells them apart?

**Method.** This notebook trains no diffusion model. For each candidate front end it:

1. trains the same small CNN classifier on real training clips (folds 1-3) and scores it on the
   validation fold (fold 4). Fold 5 is not touched;
2. measures the **ceiling**: real validation clips are converted to the representation and back to
   audio with Griffin-Lim (how generated spectrograms become audio), and compared with the
   originals.

A **negative control** uses only 2.5-5 kHz, where there are almost no calls. If it still separates
a class well, that class can be recognised from background or recording equipment rather than
from its calls.

**Decision rule, fixed before running** (thresholds in the config cell). A candidate replaces the
current front end only if:

* its F1 on the four fine-grained classes (the three groupers and squirrelfish) beats the current
  front end, with the lower end of the 95% bootstrap interval of the paired difference above 0;
* its overall macro-F1 is not clearly worse (upper end of that interval above 0);
* its ceiling is not worse: log-spectral distance below 1 kHz within `LSD_TOL_DB` of the current
  one, and accuracy on Griffin-Lim audio within `GL_ACC_TOL`.

If several pass, the one with the highest fine-grained F1 wins. If none passes, the current front
end stays.

Uncertainty comes from a **grouped bootstrap**: whole provisional recording groups are resampled
within each class, because neighbouring clips from one recording are not independent.

Runtime on a T4: about 30 minutes. Set `QUICK_TEST = True` to check it end to end first.

## 1. Setup

In [ ]:
!pip install -q diffusers accelerate soundfile
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, json, time
REPO_DIR = "/content/thesis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/1jvaldez1/audio-diffusion-synth.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
sys.path.insert(0, REPO_DIR)
import common as C

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 2. Config

The data options must match the training notebook, so the processed data saved there is reused
instead of re-read.

Candidates, and why each is here:

| Name | Range | FFT | Bands below 250 Hz / above 2 kHz | Why |
|---|---|---|---|---|
| `current` | 20-5000 Hz | 1,024 (9.8 Hz, 102 ms) | 17 / 46 | The baseline: what the diffusion model was trained with. |
| `fft2048` | 20-5000 Hz | 2,048 (4.9 Hz, 205 ms) | 17 / 46 | Same bands, finer frequency detail, coarser timing. Separates the effect of the FFT size from the effect of the range. |
| `3k_fft2048` | 20-3000 Hz | 2,048 | 21 / 24 | Moves bands down; keeps nearly all yellowfin and squirrelfish content (99% below 3.4-4.5 kHz). |
| `2k5_fft2048` | 20-2500 Hz | 2,048 | 23 / 14 | Moves bands furthest down; loses a little high content. |
| `control_hf` | 2500-5000 Hz | 1,024 | 0 / 128 | Negative control: almost no calls, mostly background. |

A 2,500 Hz range with the 1,024-point FFT is not tested: its bands near 100 Hz would be closer
together (9.2 Hz) than the FFT resolves (9.8 Hz), so the extra bands would be near-duplicates.

In [ ]:
QUICK_TEST = False

# data: must match notebooks/ocean_path_a.ipynb
SOURCE      = "/content/drive/MyDrive/thesis_runs/data/ocean_sounds_thesis.zip"
GROUP_SIZE  = 20
HIGHPASS_HZ = 20.0
NORMALIZE   = "rms"
OUT_ROOT    = "/content/drive/MyDrive/thesis_runs/ocean"

CANDIDATES = {
    "current":     dict(n_fft=1024),
    "fft2048":     dict(n_fft=2048),
    "3k_fft2048":  dict(n_fft=2048, f_max=3000.0),
    "2k5_fft2048": dict(n_fft=2048, f_max=2500.0),
}
CONTROL = {"control_hf": dict(n_fft=1024, f_min=2500.0)}
BASELINE = "current"
FINE = ["Nassau grouper", "Yellowfin grouper", "Black grouper", "Squirrelfish"]

# classifier
N_SEEDS, EPOCHS, BATCH, LR = 3, 6, 128, 2e-3

# ceiling
N_GL_PER_CLASS, GL_ITERS = 50, 64

# decision rule (pre-registered: do not change after seeing results)
N_BOOT      = 1000
LSD_TOL_DB  = 0.5       # ceiling may be at most this much worse (dB, below 1 kHz)
GL_ACC_TOL  = 0.02      # accuracy on Griffin-Lim audio may drop at most this much more than the baseline's

if QUICK_TEST:
    N_SEEDS, EPOCHS, N_GL_PER_CLASS, GL_ITERS, N_BOOT = 1, 1, 10, 16, 200

cfg = C.ocean_config(out_root=OUT_ROOT, data_root=f"{OUT_ROOT}/data")
PROBE_DIR = f"{OUT_ROOT}/probe" + ("_quick" if QUICK_TEST else "")
os.makedirs(PROBE_DIR, exist_ok=True)
C.seed_everything(cfg.seed)

## 3. Data

Each clip is center-cropped to 19,812 samples (the length that gives exactly 128 frames), the same
crop for every candidate. Training waveforms stay on the GPU in float16; spectrograms are computed
per batch, so trying a front end costs no extra memory.

In [ ]:
data = C.load_ocean(cfg, SOURCE, group_size=GROUP_SIZE, highpass_hz=HIGHPASS_HZ, normalize=NORMALIZE)
class_names = data["class_names"]
train_split, val_split, _ = C.split_by_fold(data, cfg)
del data

def center_crop(w):
    off = (w.shape[-1] - cfg.clip_samples) // 2
    return w[..., off:off + cfg.clip_samples]

train_wave = center_crop(train_split["waveforms"]).to(device)          # float16
val_wave   = center_crop(val_split["waveforms"]).to(device)
ytr, yva = train_split["labels"].to(device), val_split["labels"].numpy()
val_groups = np.array(val_split["groups"])
n_classes = len(class_names)
fine_idx = [class_names.index(n) for n in FINE]

# class-balanced draws for training: each clip weighted by 1 / its class size
counts = torch.bincount(train_split["labels"], minlength=n_classes).float()
w_train = (1.0 / counts)[train_split["labels"]]
print(f"train {len(ytr)} | val {len(yva)} clips | val groups {len(set(val_groups))}")

## 4. The probe classifier

The same network and training recipe for every front end, so the front end is the only thing that
changes. It is deliberately small and plain: the goal is to measure how much class information each
representation exposes to an ordinary classifier, not to build the best fish classifier.

| Choice | Reason |
|---|---|
| Four convolution blocks (32, 64, 128, 256 channels), each two 3x3 convolutions + batch norm + ReLU, then 2x2 max-pooling | The standard small spectrogram CNN (the VGG-style layout used in many audio-tagging baselines). 3x3 kernels see local time-frequency patterns such as harmonics, pulses and frequency sweeps; four poolings reduce 128 x 128 to 8 x 8 while channels grow to hold more pattern types. About 1.2M parameters: enough to learn the classes from 25k clips, small enough to train in about a minute. |
| Average over time, keep the frequency axis (256 channels x 8 frequency rows), then one linear layer | A call can start anywhere in the 2 s window, so time position should not matter; averaging over time removes it. Frequency position does matter, and it is the thing under test, so it is kept rather than averaged away. |
| Dropout 0.3 before the output layer, weight decay 1e-4 | Light regularisation. Neighbouring clips are near-copies, so a classifier can memorise easily. |
| No data augmentation | Augmentations such as frequency masking act differently on different band layouts and would blur the comparison. |
| Class-balanced sampling | Without it the probe would favour toadfish (8.6 times black grouper), and the fine-grained F1 would mostly measure imbalance. |
| AdamW, one-cycle learning rate, fixed 6 epochs, final weights | A fixed schedule with no early stopping: the validation fold is the evaluation set, so it must not also pick the stopping point. |
| 3 seeds, probabilities averaged | Small classifiers vary from run to run. Averaging 3 seeds reduces that noise; the spread between seeds is reported too. |
| Mixed precision (float16) on GPU | About twice as fast on a T4, no effect on the comparison. |

In [ ]:
class ProbeCNN(nn.Module):
    def __init__(self, n_classes, n_bands=128, width=(32, 64, 128, 256), p_drop=0.3):
        super().__init__()
        layers, c_in = [], 1
        for c in width:
            layers += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.MaxPool2d(2)]
            c_in = c
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Dropout(p_drop), nn.Linear(width[-1] * (n_bands // 2 ** len(width)), n_classes))

    def forward(self, x):                      # x: (B, 1, n_bands, frames)
        h = self.features(x).mean(dim=-1)      # (B, C, n_bands / 16): average over time only
        return self.head(h.flatten(1))


def make_frontend(overrides):
    fcfg = C.ocean_config(**overrides)
    fit_idx = torch.randperm(len(train_wave), generator=torch.Generator().manual_seed(0))[:4000]
    return C.MelFrontEnd(fcfg, device=device).fit(train_wave[fit_idx.to(device)].float())


def train_probe(fe, seed):
    torch.manual_seed(seed)
    model = ProbeCNN(n_classes).to(device)
    steps = EPOCHS * (len(ytr) // BATCH)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps)
    scaler = torch.amp.GradScaler(enabled=device == "cuda")
    g = torch.Generator().manual_seed(seed)
    model.train()
    for step in range(steps):
        idx = torch.multinomial(w_train, BATCH, replacement=True, generator=g).to(device)
        x = fe(train_wave[idx].float())
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
            loss = F.cross_entropy(model(x), ytr[idx])
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    return model.eval()


@torch.no_grad()
def predict(models, fe, waves, batch=256):
    """Seed-averaged class probabilities, (N, n_classes)."""
    out = []
    for i in range(0, len(waves), batch):
        x = fe(waves[i:i + batch].to(device).float())
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
            out.append(torch.stack([m(x).float().softmax(-1) for m in models]).mean(0).cpu())
    return torch.cat(out).numpy()


def f1_per_class(y, p, k=n_classes):
    cm = np.bincount(y * k + p, minlength=k * k).reshape(k, k)
    tp = np.diag(cm).astype(float)
    prec = tp / np.maximum(cm.sum(0), 1); rec = tp / np.maximum(cm.sum(1), 1)
    return np.where(prec + rec > 0, 2 * prec * rec / np.maximum(prec + rec, 1e-12), 0.0)


print(f"probe parameters: {sum(p.numel() for p in ProbeCNN(n_classes).parameters()):,}")

## 5. Train and score every front end

In [ ]:
frontends, models, val_probs, seed_scores = {}, {}, {}, {}
for name, ov in {**CANDIDATES, **CONTROL}.items():
    t0 = time.time()
    fe = make_frontend(ov); frontends[name] = fe
    models[name] = [train_probe(fe, cfg.seed + s) for s in range(N_SEEDS)]
    val_probs[name] = predict(models[name], fe, val_wave)
    seed_scores[name] = [f1_per_class(yva, predict([m], fe, val_wave).argmax(1)) for m in models[name]]
    f1 = f1_per_class(yva, val_probs[name].argmax(1))
    print(f"{name:<13} macro-F1 {f1.mean():.3f} | fine-grained F1 {f1[fine_idx].mean():.3f} | {(time.time() - t0) / 60:.1f} min")

rows = []
for name in val_probs:
    f1 = f1_per_class(yva, val_probs[name].argmax(1))
    per_seed = np.array([s[fine_idx].mean() for s in seed_scores[name]])
    rows.append({"front end": name, "macro-F1": f1.mean(), "fine-grained F1": f1[fine_idx].mean(),
                 "fine F1 seed sd": per_seed.std() if len(per_seed) > 1 else float("nan"),
                 **{n: f1[i] for i, n in enumerate(class_names)}})
scores = pd.DataFrame(rows).set_index("front end")
scores.round(3)

### Grouped bootstrap: paired differences against the current front end

Each resample draws whole recording groups with replacement within every class, and every front end
is scored on the same resample, so the differences are paired.

In [ ]:
rng = np.random.default_rng(0)
cls_groups = [[np.flatnonzero((yva == c) & (val_groups == g)) for g in np.unique(val_groups[yva == c])]
              for c in range(n_classes)]
preds = {n: p.argmax(1) for n, p in val_probs.items()}
boot = {n: [] for n in preds}
for _ in range(N_BOOT):
    idx = np.concatenate([np.concatenate([grp[j] for j in rng.integers(len(grp), size=len(grp))]) for grp in cls_groups])
    for n, p in preds.items():
        f1 = f1_per_class(yva[idx], p[idx])
        boot[n].append((f1.mean(), f1[fine_idx].mean()))
boot = {n: np.array(v) for n, v in boot.items()}

diff_rows = []
for n in CANDIDATES:
    if n == BASELINE:
        continue
    d = boot[n] - boot[BASELINE]
    diff_rows.append({"front end": n,
                      "macro-F1 diff": d[:, 0].mean(), "macro 95% CI": tuple(round(float(v), 3) for v in np.percentile(d[:, 0], [2.5, 97.5])),
                      "fine F1 diff": d[:, 1].mean(), "fine 95% CI": tuple(round(float(v), 3) for v in np.percentile(d[:, 1], [2.5, 97.5]))})
diffs = pd.DataFrame(diff_rows).set_index("front end")
diffs.round(3)

## 6. Ceiling: what survives the trip back to audio

For each candidate, `N_GL_PER_CLASS` validation clips per class are converted to its spectrogram and
back with Griffin-Lim. Two measures:

* **Log-spectral distance** (dB, lower is better) between original and reconstruction, on one
  common fine-grained STFT for every candidate, in three ranges: below 250 Hz (grouper and toadfish
  calls), 250 Hz-1 kHz (squirrelfish, vessel tonals) and 1-5 kHz.
* **Accuracy on Griffin-Lim audio**: the candidate's own classifier on the reconstructed clips,
  against the same clips' originals. A large drop means the inversion destroys what the
  classifier relies on, which generated audio would suffer too.

The 1-5 kHz distance is large by construction for front ends that stop at 2.5 or 3 kHz: that
content is removed, which is the trade being tested. The decision rule uses only the range below
1 kHz; listen to a few reconstructions if the upper range matters for a class.

In [ ]:
gl_idx = np.concatenate([np.random.default_rng(1).choice(np.flatnonzero(yva == c), N_GL_PER_CLASS, replace=False)
                         for c in range(n_classes)])
gl_real = val_wave[torch.from_numpy(gl_idx).to(device)].float()
ref_stft = lambda w: torch.stft(w.squeeze(1).to(device), n_fft=2048, hop_length=156,
                                window=torch.hann_window(2048, device=device), return_complex=True).abs() ** 2
freqs = np.fft.rfftfreq(2048, 1 / cfg.sample_rate)
BANDS = {"<250 Hz": (20, 250), "250-1000 Hz": (250, 1000), "1-5 kHz": (1000, 5000)}

def lsd(P, Q, lo, hi, eps=1e-8):
    m = torch.from_numpy((freqs >= lo) & (freqs < hi)).to(P.device)
    d = 10 * torch.log10(P[:, m] + eps) - 10 * torch.log10(Q[:, m] + eps)
    return float(torch.sqrt((d ** 2).mean(dim=1)).mean())

ceil_rows = []
P_real = ref_stft(gl_real)
for name in CANDIDATES:
    fe = frontends[name]
    gl_wave = C.spec_to_wave(fe(gl_real).cpu(), fe, n_iter=GL_ITERS)
    P_gl = ref_stft(gl_wave)
    acc_real = (predict(models[name], fe, gl_real).argmax(1) == yva[gl_idx]).mean()
    acc_gl = (predict(models[name], fe, gl_wave).argmax(1) == yva[gl_idx]).mean()
    ceil_rows.append({"front end": name, **{f"LSD {b}": lsd(P_real, P_gl, *r) for b, r in BANDS.items()},
                      "acc real": acc_real, "acc Griffin-Lim": acc_gl, "acc drop": acc_real - acc_gl})
ceiling = pd.DataFrame(ceil_rows).set_index("front end")
ceiling["LSD <1 kHz"] = (ceiling["LSD <250 Hz"] + ceiling["LSD 250-1000 Hz"]) / 2
ceiling.round(3)

## 7. Decision and negative control

In [ ]:
verdict = {}
for n in CANDIDATES:
    if n == BASELINE:
        continue
    d = boot[n] - boot[BASELINE]
    checks = {
        "fine-grained F1 better (CI above 0)": np.percentile(d[:, 1], 2.5) > 0,
        "macro-F1 not clearly worse":          np.percentile(d[:, 0], 97.5) > 0,
        "ceiling LSD within tolerance":        ceiling.loc[n, "LSD <1 kHz"] <= ceiling.loc[BASELINE, "LSD <1 kHz"] + LSD_TOL_DB,
        "Griffin-Lim accuracy within tolerance": ceiling.loc[n, "acc drop"] <= ceiling.loc[BASELINE, "acc drop"] + GL_ACC_TOL,
    }
    verdict[n] = checks
    print(f"{n}: " + ", ".join(f"{k}: {'yes' if v else 'NO'}" for k, v in checks.items()))
passing = [n for n, c in verdict.items() if all(c.values())]
winner = max(passing, key=lambda n: scores.loc[n, "fine-grained F1"]) if passing else BASELINE
print(f"\n=> front end to use: {winner}" + ("" if passing else "  (no candidate passed every check; keeping the current one)"))

print("\nNegative control (2.5-5 kHz only), per-class F1. Chance is about 0.17.")
ctrl = scores.loc["control_hf", class_names]
for n in class_names:
    flag = "  <- separable without its calls: likely background/equipment" if ctrl[n] >= 0.8 else ""
    print(f"   {n:<20}{ctrl[n]:.2f}{flag}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, name in zip(axes, [BASELINE, winner if winner != BASELINE else "2k5_fft2048", "control_hf"]):
    p = val_probs[name].argmax(1)
    cm = np.bincount(yva * n_classes + p, minlength=n_classes ** 2).reshape(n_classes, n_classes)
    cm = cm / cm.sum(1, keepdims=True)
    ax.imshow(cm, vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(n_classes), class_names, rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(n_classes), class_names, fontsize=7); ax.set_title(name)
    for i in range(n_classes):
        for j in range(n_classes):
            ax.text(j, i, f"{cm[i, j]:.0%}", ha="center", va="center", color="w", fontsize=7)
plt.suptitle("validation confusion (rows: true class)"); plt.tight_layout()
plt.savefig(f"{PROBE_DIR}/confusion.png", dpi=130); plt.show()

result = {"winner": winner, "passing": passing,
          "candidates": {**CANDIDATES, **CONTROL},
          "scores": scores.round(4).reset_index().to_dict("records"),
          "diffs": diffs.reset_index().astype(str).to_dict("records"),
          "ceiling": ceiling.round(4).reset_index().to_dict("records"),
          "verdict": {n: {k: bool(v) for k, v in c.items()} for n, c in verdict.items()},
          "settings": dict(n_seeds=N_SEEDS, epochs=EPOCHS, n_boot=N_BOOT, lsd_tol_db=LSD_TOL_DB, gl_acc_tol=GL_ACC_TOL,
                           group_size=GROUP_SIZE, highpass_hz=HIGHPASS_HZ, normalize=NORMALIZE)}
with open(f"{PROBE_DIR}/frontend_probe.json", "w") as f:
    json.dump(result, f, indent=2, default=str)
print("saved", f"{PROBE_DIR}/frontend_probe.json")